In [7]:
#!pip install langchain
#!pip install langchain-community

In [3]:
!pip show langchain

Name: langchain
Version: 1.2.10
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /Users/santhoshrishimarkonda/miniforge3/lib/python3.12/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
gpt = OpenAI(api_key=api_key)

In [11]:
policy_text = """
At TechStream, we are committed to providing the best electronics to our customers.

RETURN POLICY:
We offer a 14-day return policy for all electronics, starting from the date of delivery.
However, there is a special exception for laptops: Laptops must be returned unopened with the original seal.
If the seal on a laptop box is broken, we cannot accept the return due to security and data privacy concerns.

SHIPPING INFO:
Shipping is free for all orders over $50 within the continental United States.
For orders under $50, a flat shipping fee of $5.99 applies.
International shipping is currently not available.

SUPPORT HOURS:
For technical support, our team is available Monday to Friday, from 9:00 AM to 5:00 PM EST.
We are closed on weekends and federal holidays.
"""

with open("company_policy.txt", "w") as f:
    f.write(policy_text)

print("Knowledge base file created: company_policy.txt")


Knowledge base file created: company_policy.txt


In [12]:
# We use 'langchain_core' instead of 'langchain.chains' to avoid the error
try:
    from langchain_community.document_loaders import TextLoader
    from langchain_text_splitters import CharacterTextSplitter
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    from langchain_community.vectorstores import FAISS

    # New imports for LCEL
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.output_parsers import StrOutputParser

    print("Libraries imported successfully (Using LCEL Architecture).")

except ImportError as e:
    # The rest of this block is cut off in the screenshot
    print("Error:", e)

Libraries imported successfully (Using LCEL Architecture).


In [ ]:
def build_rag(policy_path):
    print("\n--- BUILDING THE RAG ---")

    # 1. LOAD
    loader = TextLoader(policy_path)
    docs = loader.load()

    # 2. SPLIT
    text_splitter = CharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=20
    )
    splits = text_splitter.split_documents(docs)

    # 3. INDEX (Vector Store)
    print(" Indexing data...")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = FAISS.from_documents(splits, embeddings)
    retriever = vectorstore.as_retriever()
    
    return retriever



def run_modern_bot(policy_path):
    retriever = build_rag(policy_path)
    # 4. DEFINE THE PROMPT
    # We explicitly tell the AI how to use the context
    template = """
    You are a helpful customer support assistant for TechStream.
    Use the following pieces of retrieved context to answer the question.
    If the answer is not in the context, say "I don't know" or "That is not in our policy."

    Context: {context}
    Question: {question}
    Answer:"""

    prompt = ChatPromptTemplate.from_template(template.strip())

    # 5. BUILD THE CHAIN (LCEL)
    # This replaces 'RetrievalQA' with a flexible pipeline
    llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")

    def format_docs(documents):
        return "\n\n".join(
            doc.page_content for doc in documents
        )

    # The Pipeline:
    # Retrieve Context -> Format it -> Pass to Prompt -> Pass to LLM -> Parse String
    
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    print(" Bot is Online!\n")
    
    return rag_chain



In [ ]:
if __name__ == "__main__":
    
    policy_path = './company_policy.txt'
    rag_chain = run_modern_bot(policy_path)
    
    # 6. TEST

    questions = [
        "What is the policy about?",
        "What is the return policy for laptops any timeline?",
        "Do you sell hamburgers?",
        "What are your support hours?"
    ]

    for q in questions:
        print(f"User: {q}")

        try:
            # .invoke() runs the whole pipeline
            response = rag_chain.invoke(q)
            print(f"Bot: {response}\n")

        except Exception as e:
            print(f"Error: {e}\n")
            


--- BUILDING THE RAG ---
 Indexing data...
 Bot is Online!

User: What is the policy about?
Bot: The policy is about the return of electronics purchased from TechStream.

User: What is the return policy for laptops any timeline?
Bot: Laptops must be returned unopened with the original seal.

User: Do you sell hamburgers?
Bot: That is not in our policy.

User: What are your support hours?
Bot: Our team is available Monday to Friday, from 9:00 AM to 5:00 PM EST.

User: What is the policy about?
Error: 'NoneType' object has no attribute 'invoke'

User: What is the return policy for laptops any timeline?
Error: 'NoneType' object has no attribute 'invoke'

User: Do you sell hamburgers?
Error: 'NoneType' object has no attribute 'invoke'

User: What are your support hours?
Error: 'NoneType' object has no attribute 'invoke'



In [34]:
amazon_policy = """
General Returns Policy
In the event customers are found to misuse the return policy by excessively returning, or cancelling or not accepting the orders placed, Amazon reserves the right to warn and/or suspend and/or block and/or terminate such customer accounts, as necessary.

Applicable products are returnable within the applicable return window if you've received them in a condition that is physically damaged, has missing parts or accessories, defective or different from their description on the product detail page on Amazon.in.
If you report an issue with your Smartphone, Tablet, Laptop, Television, Air Conditioner, Refrigerator, Washing Machine, Microwave, we may facilitate scheduling a technician visit to your location and services availability depends on the location pin code. This visit is scheduled by placing an order with Amazon through Amazon.in Home Services. Subject to Amazon.in Home Services Terms and Conditions, a resolution will be provided based on the technician's evaluation report.
We will carry out verification checks and return will be processed only if:
it is determined that the product was not damaged while in your possession;
the product is not different from what was shipped to you;
the product is returned in original condition (with brand’s/manufacturer's box, MRP tag intact, user manual, warranty card and all the accessories therein).
If you wish to return an electronic device that stores any personal information, please ensure that you have removed all such personal information from the device prior to returning. Amazon shall not be liable in any manner for any misuse or usage of such information.
Products may not be eligible for return in some cases as covered in detail in the “Return Policy Types” table below, including cases of buyer's remorse such as incorrect model or color of product ordered or incorrect product ordered.
Products marked as "non-returnable" on the product detail page cannot be returned. However, in an unlikely event of damaged, defective or wrong item delivered to you, we will provide a full refund or replacement, as applicable. We may contact you to ascertain the damage or defect in the product prior to issuing refund/replacement. We reserve the right to pick up the product to ascertain the damage or defect in the product prior to issuing refund/replacement.
Returns eligibility is based on your account history, including information already associated with your Amazon account such as account standing, shopping and returns activity over the past six to twelve months, and whether your account has any association with returns misuse or fraudulent activity — in accordance with our [Privacy Notice]
If Inspection Service is provided for the product(s) purchased by you (see Inspection Service Terms and Conditions[here]) and OTP is shared after verifying that product is not wrong, damaged or missing accessories, you will not be able to return the product claiming that the product is wrong, damaged, or missing accessories.
No additional information is required to return an eligible order unless otherwise noted in the category specific policy.
Products may be eligible for replacement only if the same seller has the exact same item in stock.
On products where return is eligible (basis the policy defined in the Return Policy Types table below), if the seller does not have the exact same product in stock, a refund would be issued to you instead of a Replacement.
Products purchased by international customers are not eligible for returns. However, orders made by international customers are eligible for refunds and customers will have to contact customer service within 5 business days from delivery date or estimated delivery date to claim refunds. For India based customers please refer to the display page to confirm the days of replacement / return specific to the item before purchase.
To know about the Return window for Amazon Business orders, please visit here.
If products received under the Amazon Vine program are faulty, please reach out via “Contact Us” link in the Amazon Vine Portal.
Amazon does not allow requests for unlimited returns and/or free returns. In the event customers are found to be excessively returning, or cancelling or not accepting the orders placed, Amazon reserves the right to charge a Return processing fees and /or suspend and/or block and/or terminate such customer accounts, in that order, as necessary. The Return processing fee helps cover the handling and processing costs associated with returns. The Return processing fee will be deducted from the total amount paid by the customer for the returned order, before the refund is issued, subject to satisfaction of Amazon terms and conditions of use. Amazon reserves the right to modify, waive, or update the Return processing fee, at its discretion.
Amazon will not reimburse any withholding tax to customers on Return processing fees.
You may exchange an item for a different size or color (subject to availability on the app/website) or opt for a full refund. To be eligible, items must be returned in their original condition (with the brand outer box, MRP tags attached, warranty cards, and all original accessories in manufacturer packaging).

Note: If you've received a non-returnable product in a damaged/defective condition, you can contact us within 5 days from the delivery of the product.
Note: All product categories are non-returnable for International Customers for Export Orders.
"""

with open("amazon_policy.txt", "w") as f:
    f.write(amazon_policy)

print("Knowledge base file created: amazon_policy.txt")


Knowledge base file created: amazon_policy.txt


In [35]:
if __name__ == "__main__":
    policy_path = './amazon_policy.txt'
    run_modern_bot(policy_path)

Created a chunk of size 4963, which is longer than the specified 1000



--- BUILDING THE RAG ---
 Indexing data...
 Bot is Online!

User: What is the policy about?
Bot: The policy is about the return and replacement of products purchased on Amazon.in, including eligibility criteria, conditions for return, and consequences for misuse of the return policy.

User: What is the return policy for laptops any timeline?
Bot: Laptops can be returned within the applicable return window if they are physically damaged, have missing parts or accessories, defective, or different from their description on the product detail page on Amazon.in. The return policy for laptops does not specify a specific timeline, but customers can contact Amazon within 5 days from the delivery of the product if it is non-returnable and in a damaged/defective condition.

User: Do you sell hamburgers?
Bot: That is not in our policy.

User: What are your support hours?
Bot: I don't know.

